# Maker/Taker Classification Demonstration

This notebook demonstrates how to classify orders as "maker" or "taker" based on orderbook liquidity and order size. It also shows how to train a machine learning model for this classification.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import joblib

# Add the parent directory to the path so we can import the modules
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), '..'))
from trade_simulator.src.orderbook import OrderBook
from trade_simulator.models.maker_taker import (
    classify_order_heuristic,
    extract_orderbook_features,
    train_maker_taker_model,
    classify_order_ml
)

# Set random seed for reproducibility
np.random.seed(42)

## Generate Synthetic Orderbook Data

We'll create a function to generate synthetic orderbook data with varying levels of liquidity.

In [ ]:
def generate_synthetic_orderbook(base_price=50000.0, depth=10, spread=10.0, liquidity_factor=1.0):
    """
    Generate a synthetic orderbook with specified parameters.
    
    Args:
        base_price (float): The base price for the asset
        depth (int): Number of levels to generate on each side
        spread (float): The spread between best bid and best ask
        liquidity_factor (float): Factor to scale the liquidity (higher = more liquidity)
        
    Returns:
        OrderBook: A populated OrderBook instance
    """
    book = OrderBook("BTC-USDT-SWAP")
    
    # Calculate best bid and ask prices
    best_ask = base_price + spread / 2
    best_bid = base_price - spread / 2
    
    # Generate ask levels (ascending prices)
    asks = []
    for i in range(depth):
        price = best_ask + i * 10.0  # Price increases by $10 per level
        size = (1.0 + i * 0.5) * liquidity_factor  # Size increases with price
        asks.append([str(price), str(size)])
    
    # Generate bid levels (descending prices)
    bids = []
    for i in range(depth):
        price = best_bid - i * 10.0  # Price decreases by $10 per level
        size = (1.0 + i * 0.5) * liquidity_factor  # Size increases with depth
        bids.append([str(price), str(size)])
    
    # Create a tick with the generated data
    tick = {
        "timestamp": "2023-01-01T00:00:00Z",
        "exchange": "OKX",
        "symbol": "BTC-USDT-SWAP",
        "asks": asks,
        "bids": bids
    }
    
    book.update_from_tick(tick)
    return book

## Generate Synthetic Orders

Now we'll generate synthetic orders with known maker/taker labels.

In [ ]:
def generate_synthetic_orders(book, n_orders=100):
    """
    Generate synthetic orders with known maker/taker labels.
    
    Args:
        book (OrderBook): The orderbook to use for reference
        n_orders (int): Number of orders to generate
        
    Returns:
        tuple: (orderbooks, orders) where orderbooks is a list of OrderBook instances
               and orders is a list of dictionaries with keys 'size_usd', 'side', and 'is_maker'
    """
    orderbooks = []
    orders = []
    
    # Get reference values
    mid_price = book.mid_price()
    best_ask_price, best_ask_size = book.asks[0]
    best_bid_price, best_bid_size = book.bids[0]
    
    best_ask_usd = float(best_ask_price) * float(best_ask_size)
    best_bid_usd = float(best_bid_price) * float(best_bid_size)
    
    for i in range(n_orders):
        # Create a copy of the orderbook with some random variation
        liquidity_factor = np.random.uniform(0.8, 1.2)
        new_book = generate_synthetic_orderbook(base_price=mid_price, liquidity_factor=liquidity_factor)
        orderbooks.append(new_book)
        
        # Randomly choose side
        side = np.random.choice(["buy", "sell"])
        
        # Determine if this should be a maker or taker order
        is_maker = np.random.choice([True, False])
        
        # Generate order size based on the label
        if side == "buy":
            reference_size = best_ask_usd
        else:  # sell
            reference_size = best_bid_usd
        
        if is_maker:
            # Maker orders are smaller than the best level
            size_usd = reference_size * np.random.uniform(0.1, 0.9)
        else:
            # Taker orders are larger than the best level
            size_usd = reference_size * np.random.uniform(1.1, 3.0)
        
        # Create the order
        order = {
            "size_usd": size_usd,
            "side": side,
            "is_maker": is_maker
        }
        
        orders.append(order)
    
    return orderbooks, orders

## Test the Heuristic Classification

Let's test the heuristic classification on some sample orders.

In [ ]:
# Create a sample orderbook
book = generate_synthetic_orderbook()
print(book)

# Get reference values
mid_price = book.mid_price()
best_ask_price, best_ask_size = book.asks[0]
best_bid_price, best_bid_size = book.bids[0]

best_ask_usd = float(best_ask_price) * float(best_ask_size)
best_bid_usd = float(best_bid_price) * float(best_bid_size)

print(f"Best ask level: {best_ask_size} BTC at ${best_ask_price} = ${best_ask_usd:.2f}")
print(f"Best bid level: {best_bid_size} BTC at ${best_bid_price} = ${best_bid_usd:.2f}")

# Test different order sizes
order_sizes = [best_ask_usd * 0.5, best_ask_usd * 1.0, best_ask_usd * 1.5]

for size in order_sizes:
    buy_class = classify_order_heuristic(size, book, "buy")
    sell_class = classify_order_heuristic(size, book, "sell")
    print(f"Order size: ${size:.2f}")
    print(f"  Buy classification: {buy_class}")
    print(f"  Sell classification: {sell_class}")

## Train a Machine Learning Model

Now we'll train a logistic regression model to classify orders as maker or taker.

In [ ]:
# Generate synthetic data
n_samples = 1000
orderbooks, orders = generate_synthetic_orders(book, n_samples)

# Split into training and testing sets
train_size = int(0.8 * n_samples)
train_orderbooks = orderbooks[:train_size]
train_orders = orders[:train_size]
test_orderbooks = orderbooks[train_size:]
test_orders = orders[train_size:]

# Train the model
model_path = "maker_taker_model.joblib"
model, scaler = train_maker_taker_model(train_orderbooks, train_orders, model_path)

print(f"Model trained and saved to {model_path}")

## Evaluate the Model

Let's evaluate the model on the test set.

In [ ]:
# Evaluate the model
y_true = [1 if order["is_maker"] else 0 for order in test_orders]
y_pred = []

for i, (test_book, test_order) in enumerate(zip(test_orderbooks, test_orders)):
    prediction = classify_order_ml(test_order["size_usd"], test_book, test_order["side"], model_path)
    y_pred.append(1 if prediction == "maker" else 0)

# Print classification report
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=["taker", "maker"]))

# Print confusion matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

## Compare with Heuristic

Let's compare the ML model with the heuristic approach.

In [ ]:
# Evaluate the heuristic
y_heuristic = []

for i, (test_book, test_order) in enumerate(zip(test_orderbooks, test_orders)):
    prediction = classify_order_heuristic(test_order["size_usd"], test_book, test_order["side"])
    y_heuristic.append(1 if prediction == "maker" else 0)

# Print classification report
print("Heuristic Classification Report:")
print(classification_report(y_true, y_heuristic, target_names=["taker", "maker"]))

# Print confusion matrix
cm = confusion_matrix(y_true, y_heuristic)
print("Heuristic Confusion Matrix:")
print(cm)

## Visualize Decision Boundary

Let's visualize the decision boundary of the ML model.

In [ ]:
# Extract features for visualization
X = []
y = []
relative_sizes = []
order_sizes = []

for i, (book, order) in enumerate(zip(test_orderbooks, test_orders)):
    try:
        # Get the best level based on the side
        side = order["side"]
        if side == "buy":
            best_price, best_size = book.asks[0]
        else:  # sell
            best_price, best_size = book.bids[0]
        
        # Calculate relative size
        best_level_usd = float(best_price) * float(best_size)
        relative_size = order["size_usd"] / best_level_usd
        
        relative_sizes.append(relative_size)
        order_sizes.append(order["size_usd"])
        y.append(1 if order["is_maker"] else 0)
    except Exception as e:
        print(f"Error processing sample {i}: {e}")

# Plot the data
plt.figure(figsize=(10, 6))
plt.scatter(relative_sizes, order_sizes, c=y, cmap="coolwarm", alpha=0.6)
plt.axvline(x=1.0, color="black", linestyle="--", label="Heuristic Boundary")
plt.xlabel("Relative Size (Order Size / Best Level Size)")
plt.ylabel("Order Size (USD)")
plt.title("Maker/Taker Classification")
plt.colorbar(label="Is Maker")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Feature Importance

Let's examine the importance of different features in the ML model.

In [ ]:
# Load the model
model_data = joblib.load(model_path)
model = model_data["model"]

# Get feature names
feature_names = [
    "mid_price", "spread", "spread_pct", "volume_imbalance",
    *[f"bid_volume_{i}" for i in range(5)],
    *[f"ask_volume_{i}" for i in range(5)],
    *[f"bid_distance_{i}" for i in range(5)],
    *[f"ask_distance_{i}" for i in range(5)],
    "order_size", "relative_size"
]

# Get feature importances
importances = np.abs(model.coef_[0])

# Sort features by importance
indices = np.argsort(importances)[::-1]

# Plot feature importances
plt.figure(figsize=(12, 6))
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), [feature_names[i] for i in indices], rotation=90)
plt.xlabel("Feature")
plt.ylabel("Importance")
plt.title("Feature Importance for Maker/Taker Classification")
plt.tight_layout()
plt.show()

## Conclusion

In this notebook, we've demonstrated two approaches to classifying orders as "maker" or "taker":

1. **Heuristic Approach**: A simple rule-based approach that classifies orders based on their size relative to the best level in the orderbook.
2. **Machine Learning Approach**: A more sophisticated approach that uses a logistic regression model trained on orderbook features.

The heuristic approach is simple and interpretable, but the ML approach can potentially capture more complex patterns in the data. The choice between the two depends on the specific requirements of the application.